# Advanced: Why Classical Hopfield Networks Converge and Why Hebbian Learning Works
This notebook answers a central question in associative memory: why does a classical Hopfield network settle, and when does it settle on the memory we intended?

> __Learning Objectives:__
> 
> By the end of this notebook, you should be able to:
>
> * __Derive one-step energy change:__ Show exactly where $\Delta E$ comes from, where $\Delta E$ denotes the one-step change in network energy after an asynchronous update. Track each algebraic simplification from the full energy difference to the compact local-field form.
> * __Connect descent to convergence:__ Explain why finite state space plus lower-bounded energy implies convergence in finite time. Distinguish clearly between non-increasing updates and strictly decreasing accepted flips.
> * __Interpret Hebbian recall limits:__ Derive the signal-crosstalk decomposition and relate higher load $\alpha=K/N$ to retrieval errors, where $K$ is the number of stored patterns and $N$ is the number of neurons. Use this decomposition to explain why convergence can still end at the wrong attractor.

Let's get started!
___

## Derivation

Before we write equations, fix the narrative arc: asynchronous updates create descent, and Hebbian learning shapes where that descent ends.

> __Intuition first:__
>
> Before writing formal algebra, it helps to picture the network as a system moving on an energy landscape. Each asynchronous update asks whether changing one bit moves the state downhill.
>
> * __Why convergence should happen__: If every accepted update lowers one global scalar energy, the network can only move downhill or stay flat. Because the number of binary states is finite and the energy is bounded below, descent cannot continue forever and the dynamics must eventually stop.
> * __Why memory retrieval can fail__: Stopping does not guarantee that the endpoint is the memory we intended to retrieve. Interference between stored patterns can create competing local minima, so the same descent process can converge to a spurious or wrong attractor; here, a spurious attractor means a stable state that is not one of the stored memories.
>
> Now let's turn this picture into a theorem and prove it step by step.


Before we formalize the claim, let's fix notation so each symbol has one clear meaning.

> __Notation guide (used throughout):__
>
> Neuron indices $i,j,k\in\{1,\dots,N\}$ refer to positions in the network state vector, where $N$ is the number of neurons. Memory indices $\mu\in\{1,\dots,K\}$ label stored patterns, and $\nu\in\{1,\dots,K\}$ denotes one fixed target pattern being analyzed.
>
> We write $s_i$ for the current state of neuron $i$ and $s_i'$ for its state after one asynchronous update; an __accepted flip__ means $s_i'\neq s_i$. We use a tie-preserving sign update: $\mathrm{sign}(x)=+1$ for $x>0$, $\mathrm{sign}(x)=-1$ for $x<0$, and if $x=0$ we keep the current state unchanged.
>
> The load is $\alpha=K/N$. The crosstalk term $\eta_i^{\nu}$ and stability margin $m_i^{\nu}$ are defined explicitly in the Hebbian theorem below.

With notation fixed, we can now state the formal convergence claim.

> __Convergence Theorem (Classical Hopfield, asynchronous updates):__
>
> Let $\mathbf{s}\in\{-1,+1\}^{N}$, assume $w_{ij}=w_{ji}$ and $w_{ii}=0$, and define
> $$
> \begin{align*}
> h_i(\mathbf{s}) &= \sum_{j=1}^{N} w_{ij}s_j + b_i\\
> E(\mathbf{s}) &= -\frac{1}{2}\sum_{j=1}^{N}\sum_{k=1}^{N} w_{jk}s_js_k - \sum_{j=1}^{N} b_j s_j.
> \end{align*}
> $$
> Here, $h_i(\mathbf{s})$ is the local field at neuron $i$, i.e., the weighted input plus bias evaluated at state $\mathbf{s}$. $E(\mathbf{s})$ is the global Hopfield energy at state $\mathbf{s}$, where lower values correspond to more stable configurations.
> In this notation, $w_{ij}$ is the connection weight from neuron $j$ to neuron $i$, and $b_i$ is the bias (threshold) term for neuron $i$.
> If one asynchronous update changes only neuron $i$ via $s_i' = \mathrm{sign}(h_i(\mathbf{s}))$ (and $s_j'=s_j$ for $j\neq i$), the one-step energy change is
> $$
> \Delta E = E(\mathbf{s}')-E(\mathbf{s}) = -(s_i' - s_i)h_i(\mathbf{s}).
> $$
> Using this identity, we obtain:
> * __One-step descent__: $\Delta E = E(\mathbf{s}')-E(\mathbf{s}) \le 0$. This means each asynchronous update never increases the Hopfield energy.
> * __Strict descent on flips__: if $s_i'\neq s_i$, then $\Delta E < 0$. This gives actual progress whenever the selected neuron changes state.
> * __Finite-time convergence__: repeated asynchronous updates reach a fixed point in finitely many accepted flips. Here, a fixed point means a state vector that no longer changes under subsequent asynchronous sign updates.

To prove this theorem, we derive the one-step energy change exactly.

Start from the definition and expand before simplifying:
$$
\begin{align*}
\Delta E &= E(\mathbf{s}') - E(\mathbf{s})\\
&= \left[-\frac{1}{2}\sum_{j=1}^{N}\sum_{k=1}^{N} w_{jk}s_j's_k' - \sum_{j=1}^{N} b_j s_j'\right] - \left[-\frac{1}{2}\sum_{j=1}^{N}\sum_{k=1}^{N} w_{jk}s_js_k - \sum_{j=1}^{N} b_j s_j\right]\\
&= -\frac{1}{2}\sum_{j=1}^{N}\sum_{k=1}^{N} w_{jk}s_j's_k' + \frac{1}{2}\sum_{j=1}^{N}\sum_{k=1}^{N} w_{jk}s_js_k - \sum_{j=1}^{N} b_j(s_j' - s_j)\\
&= -\frac{1}{2}\sum_{j=1}^{N}\sum_{k=1}^{N} w_{jk}(s_j's_k' - s_js_k) - \sum_{j=1}^{N} b_j(s_j' - s_j).
\end{align*}
$$

Because only index $i$ changed, all terms with $j,k\neq i$ cancel. Keep only terms involving $i$:
$$
\begin{align*}
\Delta E &= -\frac{1}{2}\sum_{k\neq i} w_{ik}(s_i's_k - s_is_k) - \frac{1}{2}\sum_{j\neq i} w_{ji}(s_js_i' - s_js_i) - b_i(s_i' - s_i)\\
&= -\frac{1}{2}(s_i' - s_i)\sum_{k\neq i} w_{ik}s_k - \frac{1}{2}(s_i' - s_i)\sum_{j\neq i} w_{ji}s_j - b_i(s_i' - s_i).
\end{align*}
$$

Use symmetry $w_{ji}=w_{ij}$ and combine the two sums:
$$
\begin{align*}
\Delta E &= -(s_i' - s_i)\sum_{j\neq i} w_{ij}s_j - b_i(s_i' - s_i)\\
&= -(s_i' - s_i)\left(\sum_{j=1}^{N} w_{ij}s_j + b_i\right)\qquad (w_{ii}=0)\\
&= -(s_i' - s_i)h_i(\mathbf{s}).
\end{align*}
$$

This identity is the engine of Hopfield convergence. To make the cancellation concrete, verify it in a small indexed case.

> __Example:__
>
> Consider neuron $i=2$ in a network with $N=4$ nodes. Starting from the definition of $\Delta E$, we have
> $$
> \begin{align*}
> \Delta E &= -\frac{1}{2}\sum_{j=1}^{4}\sum_{k=1}^{4} w_{jk}(s_j's_k' - s_js_k) - \sum_{j=1}^{4} b_j(s_j' - s_j).
> \end{align*}
> $$
>
> Only neuron 2 changes (asynchronous update), so $s_1'=s_1$, $s_3'=s_3$, and $s_4'=s_4$. The surviving terms are
> $$
> \begin{align*}
> \Delta E &= -\frac{1}{2}\Big[
> w_{21}(s_2's_1 - s_2s_1) + w_{23}(s_2's_3 - s_2s_3) + w_{24}(s_2's_4 - s_2s_4)\\
> &\qquad\quad + w_{12}(s_1s_2' - s_1s_2) + w_{32}(s_3s_2' - s_3s_2) + w_{42}(s_4s_2' - s_4s_2)\Big] - b_2(s_2' - s_2).
> \end{align*}
> $$
>
> Factor $(s_2' - s_2)$ and use symmetry of the weight matrix, $w_{jk}=w_{kj}$:
> $$
> \begin{align*}
> \Delta E &= -\frac{1}{2}(s_2' - s_2)\left[(w_{21}+w_{12})s_1 + (w_{23}+w_{32})s_3 + (w_{24}+w_{42})s_4\right] - b_2(s_2' - s_2)\\
> &= -(s_2' - s_2)(w_{21}s_1 + w_{23}s_3 + w_{24}s_4 + b_2)\\
> &= -(s_2' - s_2)h_2(\mathbf{s})\quad\blacksquare
> \end{align*}
> $$
>
> This matches the general formula exactly.

Now convert the concrete example back into the general argument and close the proof.

> __Proof conclusion:__
>
> Apply the asynchronous rule $s_i' = \mathrm{sign}(h_i(\mathbf{s}))$:
> 1. If $s_i'=s_i$, then $\Delta E=0$. This is a no-flip step, so the energy remains unchanged.
> 2. If $s_i'\neq s_i$, then $s_i'$ has the same sign as $h_i(\mathbf{s})$, so $(s_i'-s_i)h_i(\mathbf{s})>0$ and therefore $\Delta E<0$. This is the strict-descent case.
>
> Therefore updates are energy non-increasing, and accepted flips are strictly descending. Since the state space has size $2^N$ and
> $$
> E(\mathbf{s}) \ge -\frac{1}{2}\sum_{j,k}|w_{jk}| - \sum_j |b_j| > -\infty,
> $$
> strict descent cannot continue forever, so the trajectory reaches a fixed point in finitely many accepted flips.

Now we ask the second question: why does convergence often land on a stored memory?

> __Hebbian Retrieval Theorem (signal-crosstalk form):__
>
> Let $\{\boldsymbol{\xi}^{\mu}\}_{\mu=1}^{K}$ be binary memories with $\xi_i^{\mu}\in\{-1,+1\}$, and define Hebbian weights
> $$
> w_{ij}=\frac{1}{N}\sum_{\mu=1}^{K}\xi_i^{\mu}\xi_j^{\mu},\quad i\neq j,\qquad w_{ii}=0.
> $$
> For stored pattern index $\nu$, evaluate the local field at the target memory state $\mathbf{s}=\boldsymbol{\xi}^{\nu}$. We write this as $h_i := h_i(\boldsymbol{\xi}^{\nu})$ for neuron $i$, and it satisfies
> $$
> h_i = \underbrace{\frac{N-1}{N}\xi_i^{\nu}}_{\text{signal}} + \underbrace{\frac{1}{N}\sum_{\mu\neq\nu}\xi_i^{\mu}\sum_{j\neq i}\xi_j^{\mu}\xi_j^{\nu}}_{\text{crosstalk }\eta_i^{\nu}}.
> $$
> Therefore the bit-stability margin (the signed alignment between the target bit and its local field) is
> $$
> m_i^{\nu}=\xi_i^{\nu}h_i = \frac{N-1}{N}+\xi_i^{\nu}\eta_i^{\nu}.
> $$
> If $m_i^{\nu}>0$ for all $i$, then $\boldsymbol{\xi}^{\nu}$ is stable under asynchronous sign updates.

To prove this theorem, expand the local field directly and separate the $\mu=\nu$ term from the $\mu\neq\nu$ terms:
$$
\begin{align*}
h_i &= \sum_{j\neq i} w_{ij}\xi_j^{\nu}\\
&= \sum_{j\neq i}\left(\frac{1}{N}\sum_{\mu=1}^{K}\xi_i^{\mu}\xi_j^{\mu}\right)\xi_j^{\nu}\\
&= \frac{1}{N}\sum_{\mu=1}^{K}\xi_i^{\mu}\sum_{j\neq i}\xi_j^{\mu}\xi_j^{\nu}\\
&= \frac{1}{N}\xi_i^{\nu}\sum_{j\neq i}(\xi_j^{\nu})^2 + \frac{1}{N}\sum_{\mu\neq\nu}\xi_i^{\mu}\sum_{j\neq i}\xi_j^{\mu}\xi_j^{\nu}\\
&= \underbrace{\frac{N-1}{N}\xi_i^{\nu}}_{\text{signal}} + \underbrace{\frac{1}{N}\sum_{\mu\neq\nu}\xi_i^{\mu}\sum_{j\neq i}\xi_j^{\mu}\xi_j^{\nu}}_{\text{crosstalk }\eta_i^{\nu}}.
\end{align*}
$$

Then compute the margin:
$$
\begin{align*}
m_i^{\nu} &= \xi_i^{\nu}h_i\\
&= \frac{N-1}{N} + \xi_i^{\nu}\eta_i^{\nu}.
\end{align*}
$$

> __Proof conclusion:__
>
> The first term is an aligned retrieval signal from the target pattern, while the second term is interference from all other stored patterns. This decomposition explains why low load favors correct recall and high load increases the chance of spurious or wrong attractors.

___


## Numerical Verification (Julia)

Now we connect the derivation to computation and check that the behavior predicted by the math appears in simulation.

> __What we test:__
> 
> * __Trajectory-level check__: Along one asynchronous retrieval trajectory, accepted flips should never increase energy. We verify this by computing $\Delta E$ over the observed sequence of states.
> * __Capacity-level check__: As load $\alpha=K/N$ increases, recall success should decrease because crosstalk increases. We estimate this trend with repeated randomized trials at fixed corruption level.

Before running code, define the experiment parameters explicitly so each reported result has clear meaning.

> __Parameter definitions for the experiments:__ Here $N$ is the number of neurons, $K$ is the number of stored patterns, and the load is $\alpha=K/N$. A noisy cue is an initial state created by flipping a fraction of bits in one stored pattern, `trials` counts repeated randomized runs, and recall success is the fraction of trials in which the retrieved state exactly matches the target stored pattern.

First, define reusable helper functions for Hebbian storage, asynchronous retrieval, energy evaluation, and Hamming distance (the number of bit positions where two $\{-1,+1\}$ states differ).

___

In [ ]:
using Random
using LinearAlgebra
using Printf

function hebbian_weights(patterns::Matrix{Int})
    _, N = size(patterns)
    W = (patterns' * patterns) ./ N
    W[diagind(W)] .= 0.0
    return Matrix{Float64}(W)
end

function energy(state::Vector{Int}, W::Matrix{Float64}, bias::Vector{Float64}=zeros(Float64, length(state)))
    return -0.5 * dot(state, W * state) - dot(bias, state)
end

function corrupt_pattern(pattern::Vector{Int}, flip_fraction::Float64, rng::AbstractRNG)
    s = copy(pattern)
    n_flip = round(Int, flip_fraction * length(s))
    if n_flip > 0
        idx = randperm(rng, length(s))[1:n_flip]
        s[idx] .*= -1
    end
    return s
end

hamming_distance(a::Vector{Int}, b::Vector{Int}) = count(a .!= b)

function async_retrieve(initial_state::Vector{Int}, W::Matrix{Float64};
    bias::Vector{Float64}=zeros(Float64, length(initial_state)),
    max_sweeps::Int=80,
    rng::AbstractRNG=Random.default_rng())

    s = copy(initial_state)
    N = length(s)
    energies = Float64[energy(s, W, bias)]

    for _ in 1:max_sweeps
        flipped = false
        for i in randperm(rng, N)
            h_i = dot(@view(W[i, :]), s) + bias[i]
            proposed = h_i == 0 ? s[i] : (h_i > 0 ? 1 : -1)
            if proposed != s[i]
                s[i] = proposed
                flipped = true
                push!(energies, energy(s, W, bias))
            end
        end
        !flipped && break
    end

    return s, energies
end


> __Experiment 1: Monotonic energy descent__
> 
> Start from a noisy cue, run asynchronous retrieval, and track $E(\mathbf{s})$ after each accepted flip.

If the derivation is correct, the largest observed $\Delta E$ should be non-positive.

In [ ]:
rng = MersenneTwister(5820)

N = 120
K = 10
patterns = rand(rng, (-1, 1), K, N)
W = hebbian_weights(patterns)

target = vec(patterns[1, :])
noisy = corrupt_pattern(target, 0.20, rng)
retrieved, energies = async_retrieve(noisy, W; max_sweeps=80, rng=rng)

energy_diffs = diff(energies)
largest_delta_E = isempty(energy_diffs) ? 0.0 : maximum(energy_diffs)
is_monotone = all(energy_diffs .<= 1e-12)

println("Monotonic energy check (single retrieval run)")
@printf("N = %d, K = %d, alpha = %.3f\n", N, K, K / N)
println("Energy monotone non-increasing: ", is_monotone)
@printf("Largest observed delta E over accepted flips: %.3e\n", largest_delta_E)
println("Initial Hamming distance to target: ", hamming_distance(noisy, target))
println("Final Hamming distance to target:   ", hamming_distance(retrieved, target))
println("Number of accepted flips: ", length(energies) - 1)


> __Experiment 2: Recall vs. load__
> 
> Sweep $\alpha=K/N$ at fixed corruption level and estimate recall success over repeated trials.

This experiment visualizes the signal-crosstalk tradeoff from the derivation.

In [ ]:
function recall_success_rate(N::Int, alpha_values::Vector{Float64};
    trials::Int=60,
    flip_fraction::Float64=0.10,
    max_sweeps::Int=80,
    seed::Int=5821)

    rng = MersenneTwister(seed)
    rows = Tuple{Float64, Int, Float64}[]

    for alpha in alpha_values
        K = max(1, round(Int, alpha * N))
        success = 0

        for _ in 1:trials
            patterns = rand(rng, (-1, 1), K, N)
            W = hebbian_weights(patterns)
            idx = rand(rng, 1:K)
            target = vec(patterns[idx, :])
            noisy = corrupt_pattern(target, flip_fraction, rng)
            retrieved, _ = async_retrieve(noisy, W; max_sweeps=max_sweeps, rng=rng)
            retrieved == target && (success += 1)
        end

        push!(rows, (alpha, K, success / trials))
    end

    return rows
end

N = 120
alpha_values = [0.03, 0.06, 0.09, 0.12, 0.15, 0.18, 0.21]
results = recall_success_rate(N, alpha_values; trials=50, flip_fraction=0.10, max_sweeps=80, seed=5822)

println("Recall performance vs load alpha = K/N")
println("N = $(N), noise = 10% bit flips, trials per alpha = 50")
println("alpha    K    success_rate")

for (alpha, K, rate) in results
    bar = repeat("#", round(Int, 30 * rate))
    @printf("%5.2f  %3d    %6.3f  %s\n", alpha, K, rate, bar)
end


## Summary

Classical Hopfield networks converge because asynchronous single-neuron updates produce a monotone descent on a lower-bounded energy over a finite state space.

Hebbian learning makes retrieval possible through constructive signal terms, but capacity is limited because crosstalk grows with load and reshapes the attractor landscape, i.e., the set of stable states reached by asynchronous updates.

> __Key takeaways:__
>
> * __Convergence mechanism__: The line-by-line $\Delta E$ derivation shows exactly why accepted asynchronous updates cannot increase energy. This gives a formal energy-descent explanation for settling behavior.
> * __Memory mechanism__: The Hebbian field decomposition separates signal from crosstalk, which explains robust recall at low load. The stability margin makes this tradeoff explicit at the bit level.
> * __Capacity mechanism__: Increasing $\alpha=K/N$ amplifies interference, reduces stability margins, and increases retrieval errors and spurious attractors. Here, spurious attractors are stable states that are not equal to any stored pattern, which is why convergence alone does not guarantee correct retrieval.

___